In [1]:
import networkx as nx
from community import community_louvain

# -----------------------------
# métricas do grafo
# -----------------------------
def metricas(G):
    componentes = nx.number_connected_components(G)

    if G.number_of_edges() > 0:
        particao = community_louvain.best_partition(G)
        modularidade = community_louvain.modularity(particao, G)
    else:
        modularidade = 0

    return {
        "componentes": componentes,
        "modularidade": modularidade
    }

# -----------------------------
# intervenção: remover nó
# -----------------------------
def remover_no(G, no):
    G2 = G.copy()
    G2.remove_node(no)
    return G2

# -----------------------------
# efeito causal
# -----------------------------
def efeito_causal(G, no):
    base = metricas(G)
    novo = metricas(remover_no(G, no))

    return {
        "no": no,
        "delta_componentes": novo["componentes"] - base["componentes"],
        "delta_modularidade": novo["modularidade"] - base["modularidade"]
    }

# -----------------------------
# ranking causal
# -----------------------------
def ranking_causal(G):
    resultados = []

    for no in G.nodes():
        resultados.append(efeito_causal(G, no))

    for r in resultados:
        r["score"] = abs(r["delta_componentes"]) * 2 + abs(r["delta_modularidade"])

    return sorted(resultados, key=lambda x: x["score"], reverse=True)


# -----------------------------
# exemplo
# -----------------------------
G = nx.erdos_renyi_graph(50, 0.05, seed=42)

ranking = ranking_causal(G)

for r in ranking[:10]:
    print(r)

{'no': 6, 'delta_componentes': 2, 'delta_modularidade': -0.012832290087534837, 'score': 4.012832290087535}
{'no': 40, 'delta_componentes': 2, 'delta_modularidade': 0.010734587142262608, 'score': 4.010734587142263}
{'no': 0, 'delta_componentes': 1, 'delta_modularidade': 0.04817685950413231, 'score': 2.0481768595041325}
{'no': 2, 'delta_componentes': 1, 'delta_modularidade': 0.02068409797884485, 'score': 2.0206840979788447}
{'no': 23, 'delta_componentes': 1, 'delta_modularidade': -0.019570801055638687, 'score': 2.0195708010556386}
{'no': 12, 'delta_componentes': 1, 'delta_modularidade': -0.01870616583649365, 'score': 2.0187061658364938}
{'no': 13, 'delta_componentes': 1, 'delta_modularidade': 0.01629988018974038, 'score': 2.0162998801897403}
{'no': 32, 'delta_componentes': 1, 'delta_modularidade': 0.013217548241140742, 'score': 2.0132175482411405}
{'no': 18, 'delta_componentes': 1, 'delta_modularidade': -0.009780604844522811, 'score': 2.009780604844523}
{'no': 43, 'delta_componentes': -1